# LLaMA-2-7B FP16 Weight and Activation Distribution

Profile the original FP16 `meta-llama/Llama-2-7b-hf` checkpoint without quantization. The notebook records weight and activation data for the first, middle, and last `mlp.gate_proj` modules and writes JSON only; it does not import plotting libraries or render figures.

The captured activation is the exact `post_attention_layernorm` output consumed by `mlp.gate_proj`, matching the LayerNorm-to-first-FFN-linear observation point used by SmoothQuant. Activation rows correspond to tokens and weight rows correspond to deterministically sampled output channels. Both surfaces retain all 4096 input channels; exact tensor-wise and full-channel statistics are stored separately.

In [ ]:
%pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import json
import math
import os
import platform
import zipfile
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import datasets
import torch
import transformers
from datasets import load_dataset
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
EXPECTED_MODEL_REVISION = "01c7f73d771dfac7d292323805ebc428287df4f9"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
EXPECTED_SOURCE_TOKENS = 341_469
PROFILE_TOKENS = 256
INPUT_CHANNEL_STRIDE = 1
WEIGHT_OUTPUT_SAMPLES = 256
OUTPUT_DIR = Path("result")
OUTPUT_PATH = OUTPUT_DIR / "llama2-7b-fp16-distribution.json"
ARCHIVE_PATH = Path("llama2-7b-fp16-distribution.zip")

if not torch.cuda.is_available():
    raise RuntimeError("This profiler requires an NVIDIA CUDA GPU.")
if PROFILE_TOKENS <= 0:
    raise ValueError("PROFILE_TOKENS must be positive.")
if INPUT_CHANNEL_STRIDE != 1:
    raise ValueError("INPUT_CHANNEL_STRIDE must be 1 for complete-channel profiling.")
if WEIGHT_OUTPUT_SAMPLES <= 0:
    raise ValueError("WEIGHT_OUTPUT_SAMPLES must be positive.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(1234)
torch.backends.cuda.matmul.allow_tf32 = False

## Load the pinned FP16 baseline

Accept the LLaMA-2 license on Hugging Face and provide `HF_TOKEN`. The checkpoint revision is pinned to the revision used by the FP16 PPL and zero-shot baselines.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = getpass("HF_TOKEN: ")

resolved_model_revision = model_info(MODEL_ID, token=token).sha
if resolved_model_revision != EXPECTED_MODEL_REVISION:
    raise RuntimeError(
        "Model revision differs from the baseline: "
        f"{resolved_model_revision} != {EXPECTED_MODEL_REVISION}"
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=resolved_model_revision, token=token
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=resolved_model_revision,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False

parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
parameter_devices = {p.device.type for p in model.parameters()}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
assert parameter_devices == {"cuda"}, parameter_devices
assert not getattr(model, "is_quantized", False)
assert model.config.num_hidden_layers == 32
assert model.config.hidden_size == 4096
assert model.config.intermediate_size == 11008

layer_indices = [0, model.config.num_hidden_layers // 2, model.config.num_hidden_layers - 1]
assert layer_indices == [0, 16, 31]
print(f"Model revision: {resolved_model_revision}")
print(f"Profile layers: {layer_indices}")

## Prepare the deterministic token sample

Use the first 256 tokens from the same concatenated WikiText-2 test stream as the FP16 PPL baseline. Token IDs and tokenizer tokens are stored in the output so the activation surface can be reproduced and annotated later.

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
all_input_ids = tokenizer(text, return_tensors="pt").input_ids
if all_input_ids.numel() != EXPECTED_SOURCE_TOKENS:
    raise RuntimeError(
        f"Token closure failed: {all_input_ids.numel()} != {EXPECTED_SOURCE_TOKENS}"
    )
profile_input_ids = all_input_ids[:, :PROFILE_TOKENS].contiguous()
profile_token_ids = profile_input_ids[0].tolist()
profile_token_strings = tokenizer.convert_ids_to_tokens(profile_token_ids)
assert profile_input_ids.shape == (1, PROFILE_TOKENS)
print(f"Source tokens: {all_input_ids.numel():,}")
print(f"Profile tokens: {profile_input_ids.numel():,}")

## Capture post-RMSNorm `mlp.gate_proj` inputs

A forward pre-hook captures the signed FP16 tensor consumed by each selected `mlp.gate_proj`. In LLaMA-2 this is the output of `post_attention_layernorm`, before the first FFN projections and SwiGLU. Only one inference pass is required; no PPL or zero-shot scoring is repeated.

In [ ]:
captured_activations = {}
hook_handles = []

def make_capture_hook(layer_index):
    def capture_hook(module, inputs):
        if layer_index in captured_activations:
            raise RuntimeError(f"Layer {layer_index} was captured more than once.")
        activation = inputs[0].detach().reshape(-1, inputs[0].shape[-1])
        if activation.size(0) != PROFILE_TOKENS:
            raise RuntimeError(
                f"Unexpected token count at layer {layer_index}: {activation.size(0)}"
            )
        captured_activations[layer_index] = activation.to(
            device="cpu", dtype=torch.float16
        ).contiguous()
    return capture_hook

for layer_index in layer_indices:
    module = model.model.layers[layer_index].mlp.gate_proj
    hook_handles.append(module.register_forward_pre_hook(make_capture_hook(layer_index)))

try:
    with torch.inference_mode():
        model(profile_input_ids.to(model.device), use_cache=False)
finally:
    for handle in hook_handles:
        handle.remove()

if sorted(captured_activations) != layer_indices:
    raise RuntimeError(
        f"Activation capture closure failed: {sorted(captured_activations)}"
    )
for layer_index, activation in captured_activations.items():
    assert activation.shape == (PROFILE_TOKENS, model.config.hidden_size)
    assert activation.dtype == torch.float16
    assert torch.isfinite(activation).all()
    print(f"Layer {layer_index}: activation {tuple(activation.shape)}")

## Build JSON records

Surface values remain signed; the future plotting script can apply `abs()` for a magnitude plot. Channel statistics use absolute magnitude. The three-sigma threshold is defined independently for every full input channel as `mean(abs(x)) + 3 * std(abs(x))`.

In [ ]:
def empirical_quantile(values, q):
    if values.ndim != 2:
        raise ValueError("values must have shape [observations, channels].")
    k = max(1, min(values.size(0), math.ceil(q * values.size(0))))
    return values.kthvalue(k, dim=0).values


def empirical_scalar_quantile(values, q):
    flattened = values.reshape(-1)
    k = max(1, min(flattened.numel(), math.ceil(q * flattened.numel())))
    return flattened.kthvalue(k).values


def full_statistics(values):
    values_f32 = values.float()
    abs_values = values_f32.abs()
    mean_abs = abs_values.mean(dim=0)
    std_abs = abs_values.std(dim=0, unbiased=False)
    threshold = mean_abs + 3.0 * std_abs
    outlier_count = (abs_values > threshold.unsqueeze(0)).sum(dim=0)
    observation_count = values.size(0)
    channel_statistics = {
        "definition": "statistics over observations for each full input channel",
        "observation_count": observation_count,
        "input_channel_count": values.size(1),
        "signed_mean": values_f32.mean(dim=0).cpu().tolist(),
        "signed_std": values_f32.std(dim=0, unbiased=False).cpu().tolist(),
        "mean_abs": mean_abs.cpu().tolist(),
        "std_abs": std_abs.cpu().tolist(),
        "max_abs": abs_values.amax(dim=0).cpu().tolist(),
        "p99_abs": empirical_quantile(abs_values, 0.99).cpu().tolist(),
        "p999_abs": empirical_quantile(abs_values, 0.999).cpu().tolist(),
        "three_sigma_abs_threshold": threshold.cpu().tolist(),
        "three_sigma_outlier_count": outlier_count.cpu().tolist(),
        "three_sigma_outlier_fraction": (
            outlier_count.float() / observation_count
        ).cpu().tolist(),
    }

    tensor_mean_abs = abs_values.mean()
    tensor_std_abs = abs_values.std(unbiased=False)
    tensor_threshold = tensor_mean_abs + 3.0 * tensor_std_abs
    tensor_outlier_count = (abs_values > tensor_threshold).sum()
    tensor_statistics = {
        "definition": "statistics over every element of the complete captured tensor",
        "element_count": values.numel(),
        "signed_mean": values_f32.mean().item(),
        "signed_std": values_f32.std(unbiased=False).item(),
        "mean_abs": tensor_mean_abs.item(),
        "std_abs": tensor_std_abs.item(),
        "max_abs": abs_values.amax().item(),
        "p99_abs": empirical_scalar_quantile(abs_values, 0.99).item(),
        "p999_abs": empirical_scalar_quantile(abs_values, 0.999).item(),
        "three_sigma_abs_threshold": tensor_threshold.item(),
        "three_sigma_outlier_count": tensor_outlier_count.item(),
        "three_sigma_outlier_fraction": (
            tensor_outlier_count.float() / values.numel()
        ).item(),
    }
    return channel_statistics, tensor_statistics


def evenly_spaced_indices(length, count, device):
    count = min(length, count)
    indices = torch.linspace(0, length - 1, steps=count, device=device)
    indices = indices.round().to(torch.long).unique(sorted=True)
    if indices.numel() != count:
        raise RuntimeError("Evenly spaced index generation produced duplicates.")
    return indices


layer_records = []
input_channel_indices = torch.arange(
    0, model.config.hidden_size, INPUT_CHANNEL_STRIDE, dtype=torch.long
)

for layer_index in layer_indices:
    layer_name = f"model.layers.{layer_index}.mlp.gate_proj"
    module = model.model.layers[layer_index].mlp.gate_proj
    weight = module.weight.detach()
    activation = captured_activations[layer_index]
    if weight.shape != (model.config.intermediate_size, model.config.hidden_size):
        raise RuntimeError(f"Unexpected weight shape at {layer_name}: {weight.shape}")

    output_channel_indices = evenly_spaced_indices(
        weight.size(0), WEIGHT_OUTPUT_SAMPLES, weight.device
    )
    input_indices_gpu = input_channel_indices.to(weight.device)
    weight_surface = weight.index_select(0, output_channel_indices).index_select(
        1, input_indices_gpu
    ).to(device="cpu", dtype=torch.float16)
    activation_surface = activation.index_select(1, input_channel_indices)

    weight_channel_stats, weight_tensor_stats = full_statistics(weight)
    activation_channel_stats, activation_tensor_stats = full_statistics(activation)
    assert weight_channel_stats["input_channel_count"] == model.config.hidden_size
    assert activation_channel_stats["input_channel_count"] == model.config.hidden_size
    assert weight_tensor_stats["element_count"] == weight.numel()
    assert activation_tensor_stats["element_count"] == activation.numel()
    assert weight_tensor_stats["p99_abs"] > 0.0
    assert activation_tensor_stats["p99_abs"] > 0.0
    assert weight_surface.shape == (WEIGHT_OUTPUT_SAMPLES, input_channel_indices.numel())
    assert activation_surface.shape == (PROFILE_TOKENS, input_channel_indices.numel())

    layer_records.append({
        "layer_index": layer_index,
        "layer_name": layer_name,
        "weight_shape": list(weight.shape),
        "activation_shape": list(activation.shape),
        "common_input_channel_indices": input_channel_indices.tolist(),
        "weight_surface": {
            "row_axis": "output_channel",
            "column_axis": "input_channel",
            "output_channel_indices": output_channel_indices.cpu().tolist(),
            "input_channel_indices": input_channel_indices.tolist(),
            "values_dtype": "float16 serialized as JSON numbers",
            "values_signed": True,
            "values": weight_surface.tolist(),
        },
        "activation_surface": {
            "row_axis": "token_position",
            "column_axis": "input_channel",
            "token_positions": list(range(PROFILE_TOKENS)),
            "input_channel_indices": input_channel_indices.tolist(),
            "values_dtype": "float16 serialized as JSON numbers",
            "values_signed": True,
            "values": activation_surface.tolist(),
        },
        "weight_full_channel_statistics": weight_channel_stats,
        "activation_full_channel_statistics": activation_channel_stats,
        "weight_tensor_statistics": weight_tensor_stats,
        "activation_tensor_statistics": activation_tensor_stats,
    })
    print(
        f"Prepared layer {layer_index}: weight surface {tuple(weight_surface.shape)}, "
        f"activation surface {tuple(activation_surface.shape)}"
    )


## Export JSON

The JSON is the source artifact. A ZIP copy is created only to make downloading the large JSON more reliable.

In [ ]:
result = {
    "schema_version": 3,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "fp16_weight_activation_distribution",
    "model": MODEL_ID,
    "model_revision": resolved_model_revision,
    "dtype": "float16",
    "quantized": False,
    "attention_implementation": "eager",
    "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
    "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
    "split": SPLIT,
    "source_token_count": int(all_input_ids.numel()),
    "profile": {
        "module": "mlp.gate_proj",
        "activation_location": "post_attention_layernorm output / mlp.gate_proj input",
        "smoothquant_alignment": "post-normalization input to the first FFN linear projection",
        "layer_selection": "first, middle, last",
        "layer_indices": layer_indices,
        "profile_tokens": PROFILE_TOKENS,
        "input_channel_stride": INPUT_CHANNEL_STRIDE,
        "sampled_input_channels": int(input_channel_indices.numel()),
        "complete_input_channels_retained": True,
        "weight_output_samples": WEIGHT_OUTPUT_SAMPLES,
        "surface_values": "signed FP16; apply abs() for magnitude plots",
        "statistics_scope": "full tensor plus all input channels",
        "inference_passes": 1,
        "ppl_or_zero_shot_recomputed": False,
    },
    "tokens": {
        "positions": list(range(PROFILE_TOKENS)),
        "token_ids": profile_token_ids,
        "tokenizer_tokens": profile_token_strings,
    },
    "layers": layer_records,
    "validation": {
        "expected_model_revision": EXPECTED_MODEL_REVISION,
        "expected_source_tokens": EXPECTED_SOURCE_TOKENS,
        "expected_layer_indices": [0, 16, 31],
        "activation_capture_closure_passed": True,
        "surface_shape_closure_passed": True,
        "tensor_statistics_closure_passed": True,
        "all_values_finite": True,
    },
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
    },
}

with OUTPUT_PATH.open("w", encoding="utf-8") as output_file:
    json.dump(result, output_file, ensure_ascii=False, separators=(",", ":"))

with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUTPUT_PATH, arcname=OUTPUT_PATH.name)

print(f"Saved JSON: {OUTPUT_PATH.resolve()}")
print(f"JSON size: {OUTPUT_PATH.stat().st_size / 2**20:.2f} MiB")
print(f"Saved archive: {ARCHIVE_PATH.resolve()}")
print(f"Archive size: {ARCHIVE_PATH.stat().st_size / 2**20:.2f} MiB")

In [ ]:
from google.colab import files
files.download(str(ARCHIVE_PATH))